# Aiscern Audio Detector — Fine-Tuning
**Platform:** Kaggle (T4 GPU)  
**Target:** `saghi776/aiscern-audio-detector`  
**Base model:** `facebook/wav2vec2-base`  
**Expected accuracy:** >84% (EER <0.10)  
**Runtime:** ~90 minutes

### Before running:
1. Set **Accelerator → GPU T4 x2**
2. Add secret `HF_TOKEN` in Kaggle → Add-ons → Secrets
3. Create HF repo: https://huggingface.co/new?owner=saghi776&name=aiscern-audio-detector

In [ ]:
# CELL 1 — Install
!pip install -q transformers datasets accelerate huggingface_hub librosa soundfile scikit-learn

In [ ]:
# CELL 2 — Authenticate
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

secrets  = UserSecretsClient()
HF_TOKEN = secrets.get_secret('HF_TOKEN')
login(token=HF_TOKEN, add_to_git_credential=False)
print('✅ Authenticated')

In [ ]:
# CELL 3 — Configuration
BASE_MODEL    = 'facebook/wav2vec2-base'
REPO_ID       = 'saghi776/aiscern-audio-detector'
SAMPLING_RATE = 16000   # wav2vec2 requires 16kHz
MAX_DURATION  = 5       # seconds — truncate/pad all clips
TARGET_LENGTH = MAX_DURATION * SAMPLING_RATE  # 80,000 samples
BATCH_SIZE    = 8       # audio is memory-heavy
EPOCHS        = 4
LR            = 1e-4
SEED          = 42
MAX_SAMPLES   = 10000   # 5K real + 5K fake — fits in 90 min

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# CELL 4 — Load ASVspoof2019 LA (gold standard for TTS detection)
# bonafide = real human speech (label 0 = REAL)
# spoof    = TTS / voice clone  (label 1 = FAKE)
from datasets import load_dataset, concatenate_datasets
import numpy as np

print('Loading ASVspoof2019 LA dataset...')
try:
    asvspoof = load_dataset(
        'DynamicSuperb/AudioDeepfakeDetection_ASVspoof2019LA',
        split='train',
        trust_remote_code=True
    )
    print(f'ASVspoof2019: {len(asvspoof):,} samples')
    print(f'Columns: {asvspoof.column_names}')
    print(f'Sample label: {asvspoof[0].get("label") or asvspoof[0].get("class_label")}')
    HAS_ASVSPOOF = True
except Exception as e:
    print(f'ASVspoof failed: {e}')
    HAS_ASVSPOOF = False

In [ ]:
# CELL 5 — Fallback: WaveFake (if ASVspoof unavailable)
# WaveFake: 7 GAN vocoders, all fake
from datasets import load_dataset, concatenate_datasets, Audio

all_splits = []

def normalise_label(val):
    """Map any label format → 0=REAL, 1=FAKE"""
    s = str(val).lower().strip()
    if s in ('spoof', 'fake', 'ai', '1', 'generated', 'synthetic', 'bonafide_spoof'): return 1
    if s in ('bonafide', 'genuine', 'real', 'human', '0'): return 0
    return -1

# Dataset 1: ASVspoof2019
if HAS_ASVSPOOF:
    ds = asvspoof.cast_column('audio', Audio(sampling_rate=SAMPLING_RATE))
    label_col = next((c for c in ds.column_names if 'label' in c.lower() or 'class' in c.lower()), None)
    if label_col:
        ds = ds.map(lambda x: {'label': normalise_label(x[label_col])})
        ds = ds.filter(lambda x: x['label'] != -1)
        ds = ds.select_columns(['audio', 'label'])
        all_splits.append(ds)
        print(f'ASVspoof: {len(ds):,} samples added')

# Dataset 2: WaveFake (all fake)
try:
    wavefake = load_dataset('balt0/WaveFake', split='train')
    wavefake = wavefake.cast_column('audio', Audio(sampling_rate=SAMPLING_RATE))
    wavefake = wavefake.map(lambda x: {'label': 1})
    wavefake = wavefake.select_columns(['audio', 'label'])
    all_splits.append(wavefake)
    print(f'WaveFake: {len(wavefake):,} fake samples added')
except Exception as e:
    print(f'WaveFake skipped: {e}')

# Dataset 3: CommonVoice (real speech)
try:
    cv = load_dataset('mozilla-foundation/common_voice_11_0', 'en',
                      split='train', token=HF_TOKEN, trust_remote_code=True)
    cv = cv.cast_column('audio', Audio(sampling_rate=SAMPLING_RATE))
    cv = cv.map(lambda x: {'label': 0})
    cv = cv.select_columns(['audio', 'label'])
    cv = cv.shuffle(SEED).select(range(min(5000, len(cv))))
    all_splits.append(cv)
    print(f'CommonVoice: {len(cv):,} real samples added')
except Exception as e:
    print(f'CommonVoice skipped: {e}')

if not all_splits:
    raise RuntimeError('No datasets loaded — check HF_TOKEN and internet')

combined = concatenate_datasets(all_splits)
print(f'\nTotal combined: {len(combined):,}')

In [ ]:
# CELL 6 — Balance + split
combined = combined.shuffle(SEED)
real = combined.filter(lambda x: x['label'] == 0)
fake = combined.filter(lambda x: x['label'] == 1)

n = min(len(real), len(fake), MAX_SAMPLES // 2)
balanced = concatenate_datasets([
    real.shuffle(SEED).select(range(n)),
    fake.shuffle(SEED).select(range(n)),
]).shuffle(SEED)

split    = balanced.train_test_split(test_size=0.15, seed=SEED)
train_ds = split['train']
val_test = split['test'].train_test_split(test_size=0.5, seed=SEED)
val_ds   = val_test['train']
test_ds  = val_test['test']

print(f'Train: {len(train_ds):,}  Val: {len(val_ds):,}  Test: {len(test_ds):,}')

In [ ]:
# CELL 7 — Audio preprocessing
import numpy as np

def preprocess_audio(examples):
    arrays = []
    for audio in examples['audio']:
        arr = np.array(audio['array'], dtype=np.float32)
        # Normalize amplitude
        max_val = np.max(np.abs(arr))
        if max_val > 0:
            arr = arr / max_val
        # Pad or truncate to TARGET_LENGTH
        if len(arr) < TARGET_LENGTH:
            arr = np.pad(arr, (0, TARGET_LENGTH - len(arr)))
        else:
            arr = arr[:TARGET_LENGTH]
        arrays.append(arr)
    return {
        'input_values': arrays,
        'labels':       examples['label'],
    }

print('Preprocessing audio...')
train_ds = train_ds.map(preprocess_audio, batched=True, batch_size=64,
                         remove_columns=['audio'], desc='Train')
val_ds   = val_ds.map(preprocess_audio, batched=True, batch_size=64,
                       remove_columns=['audio'], desc='Val')
test_ds  = test_ds.map(preprocess_audio, batched=True, batch_size=64,
                        remove_columns=['audio'], desc='Test')

train_ds.set_format('torch')
val_ds.set_format('torch')
test_ds.set_format('torch')
print('✅ Audio preprocessing complete')

In [ ]:
# CELL 8 — Load wav2vec2 feature extractor + model
from transformers import (
    Wav2Vec2FeatureExtractor,
    Wav2Vec2ForSequenceClassification,
)

feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(BASE_MODEL)

# CRITICAL id2label for hf-analyze.ts:
# /fake|spoof|label_1|deepfake|synthetic|ai/i → matches 'FAKE'
# /real|bonafide|label_0|authentic|human/i    → matches 'REAL'
model = Wav2Vec2ForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=2,
    id2label={0: 'REAL', 1: 'FAKE'},
    label2id={'REAL': 0, 'FAKE': 1},
)

# Freeze CNN feature extractor — prevents overfitting, 40% speedup
model.freeze_feature_encoder()

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable params: {trainable / 1e6:.1f}M (CNN frozen)')

In [ ]:
# CELL 9 — Data collator
from dataclasses import dataclass
from typing import Dict, List
import torch

@dataclass
class AudioDataCollator:
    feature_extractor: Wav2Vec2FeatureExtractor

    def __call__(self, features: List[Dict]) -> Dict[str, torch.Tensor]:
        input_values = [{'input_values': f['input_values']} for f in features]
        labels       = [f['labels'] for f in features]
        batch        = self.feature_extractor.pad(
            input_values, return_tensors='pt', padding=True
        )
        batch['labels'] = torch.tensor(labels, dtype=torch.long)
        return batch

collator = AudioDataCollator(feature_extractor=feature_extractor)
print('✅ Data collator ready')

In [ ]:
# CELL 10 — Metrics (includes EER — standard ASVspoof metric)
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, roc_curve

def compute_eer(labels, scores):
    """Equal Error Rate — lower is better. Industry standard for spoof detection."""
    try:
        fpr, tpr, _ = roc_curve(labels, scores)
        fnr = 1 - tpr
        idx = np.nanargmin(np.abs(fnr - fpr))
        return float((fpr[idx] + fnr[idx]) / 2)
    except Exception:
        return 0.5

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()
    preds = (probs > 0.5).astype(int)
    return {
        'accuracy': float(accuracy_score(labels, preds)),
        'f1':       float(f1_score(labels, preds, average='binary')),
        'auc':      float(roc_auc_score(labels, probs)) if len(np.unique(labels)) > 1 else 0.5,
        'eer':      compute_eer(labels, probs),
    }

In [ ]:
# CELL 11 — Training
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback

training_args = TrainingArguments(
    output_dir                  = './aiscern-audio-detector',
    num_train_epochs            = EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE,
    learning_rate               = LR,
    warmup_ratio                = 0.1,
    evaluation_strategy         = 'epoch',
    save_strategy               = 'epoch',
    save_total_limit            = 2,
    load_best_model_at_end      = True,
    metric_for_best_model       = 'f1',
    greater_is_better           = True,
    fp16                        = True,
    gradient_accumulation_steps = 4,   # effective batch = 32
    push_to_hub                 = True,
    hub_model_id                = REPO_ID,
    hub_token                   = HF_TOKEN,
    hub_strategy                = 'every_save',
    report_to                   = 'none',
    seed                        = SEED,
)

trainer = Trainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_ds,
    eval_dataset    = val_ds,
    data_collator   = collator,
    compute_metrics = compute_metrics,
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=2)],
)

print('Starting training...')
trainer.train()
print('✅ Training complete')

In [ ]:
# CELL 12 — Evaluate on test set
test_results = trainer.evaluate(test_ds)

print(f"\n{'='*40}")
print('TEST RESULTS')
print(f"{'='*40}")
print(f"Accuracy: {test_results['eval_accuracy']:.4f} ({test_results['eval_accuracy']*100:.1f}%)")
print(f"F1:       {test_results['eval_f1']:.4f}")
print(f"AUC:      {test_results.get('eval_auc', 'N/A')}")
print(f"EER:      {test_results.get('eval_eer', 'N/A')} (lower = better, <0.10 = excellent)")

acc_ok = test_results['eval_accuracy'] >= 0.84
eer_ok = test_results.get('eval_eer', 1.0) < 0.10
print(f'\nAccuracy target (>84%): {"✅" if acc_ok else "⚠️ "}')
print(f'EER target (<0.10):     {"✅" if eer_ok else "⚠️ "}')

In [ ]:
# CELL 13 — Push to HuggingFace
acc = test_results['eval_accuracy']
eer = test_results.get('eval_eer', 'N/A')

trainer.push_to_hub(
    commit_message=f'wav2vec2 TTS detector — acc={acc:.3f} EER={eer} | Aiscern audio detector'
)
feature_extractor.push_to_hub(REPO_ID, token=HF_TOKEN)
print(f'✅ Pushed to https://huggingface.co/{REPO_ID}')
print('Note: model must return [{"label": "FAKE", ...}, {"label": "REAL", ...}]')
print('hf-analyze.ts looks for /fake/i → FAKE and /real/i → REAL')